# AI-Based Competition Generator

This notebook walks through the full pipeline:
1. Scrape a competitor's website
2. Extract links and page content
3. Feed content to Claude to generate a competitive intelligence report
4. Save the report to `outputs/`

## 1. Setup

In [1]:
# Install dependencies if needed
# !pip install requests beautifulsoup4 anthropic python-dotenv

In [2]:
import os
import anthropic
from pathlib import Path
from dotenv import load_dotenv

from scraper import fetch_website_links, fetch_website_contents
from prompts import COMPETITION_ANALYSIS_PROMPT

load_dotenv()

ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")
assert ANTHROPIC_API_KEY, "Set ANTHROPIC_API_KEY in your .env file"

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
print("Client ready.")

ModuleNotFoundError: No module named 'anthropic'

## 2. Configure Target URL

In [ ]:
# Set the competitor URL to analyse
TARGET_URL = "https://example.com"  # <-- change this

## 3. Scrape Links

In [ ]:
links = fetch_website_links(TARGET_URL)

print(f"Found {len(links)} links on {TARGET_URL}")
for link in links[:10]:
    print(" ", link)

## 4. Scrape Page Content

In [ ]:
content = fetch_website_contents(TARGET_URL)

print(f"Extracted {len(content.splitlines())} lines of content")
print("\n--- Preview (first 20 lines) ---")
print("\n".join(content.splitlines()[:20]))

## 5. Generate Competitive Analysis with Claude

In [ ]:
prompt = COMPETITION_ANALYSIS_PROMPT.format(
    url=TARGET_URL,
    links="\n".join(links),
    content=content,
)

message = client.messages.create(
    model="claude-opus-4-6",
    max_tokens=2048,
    messages=[
        {"role": "user", "content": prompt}
    ],
)

report = message.content[0].text
print(report)

## 6. Save Report

In [ ]:
from datetime import datetime

outputs_dir = Path("outputs")
outputs_dir.mkdir(exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
domain = TARGET_URL.replace("https://", "").replace("http://", "").split("/")[0]
filename = outputs_dir / f"{domain}_{timestamp}.md"

filename.write_text(report)
print(f"Report saved to: {filename}")